# Summary Figure — Predicting Ocean pCO₂ from Satellite Data

This notebook generates a single **6-panel summary figure** that tells the full story of the project,
from geographic context through data exploration to ML results.

**Panels:**
- **(A)** Map of the 7 NOAA buoy sites
- **(B)** SST vs pCO₂ scatter — the core physical relationship
- **(C)** Data availability timeline by site
- **(D)** Model comparison (RMSE & R²)
- **(E)** Predicted vs Actual for the best model
- **(F)** Feature importance from Random Forest

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sns.set_theme(style="whitegrid", font_scale=1.0)
print("Ready.")

Ready.


In [16]:
PROJECT_ROOT = Path(r"C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\ML2026_Orrand")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PLOT_DIR = PROJECT_ROOT / "final project deliverables"

In [17]:
# ──────────────────────────────────────────────
# Load all data needed for the 6 panels
# ──────────────────────────────────────────────

# 1) ML-ready training data (for panels A, B, D, E, F)
df = pd.read_csv(PROCESSED_DIR / "training_data_700_ml_ready.csv")
df["date"] = pd.to_datetime(df["date"])

# 2) Continuous data periods (for panel C — data availability)
periods = pd.read_csv(PROCESSED_DIR / "buoy_continuous_data_periods.csv")
periods["segment_start"] = pd.to_datetime(periods["segment_start"])
periods["segment_end"]   = pd.to_datetime(periods["segment_end"])

# Quick reference: one lat/lon per site
site_coords = (
    df.groupby("location")[["latitude", "longitude"]]
    .median()
    .reset_index()
)

print(f"Training data: {len(df)} rows, {df['location'].nunique()} sites")
print(f"Data periods:  {len(periods)} continuous segments")
print()
print(site_coords.to_string(index=False))

Training data: 479 rows, 7 sites
Data periods:  35 continuous segments

           location  latitude  longitude
      First Landing    36.998    -76.088
 Grays Reef Georgia    31.399    -80.869
            LA Buoy    28.869    -90.480
            La Push    47.964   -124.958
      SE Bering Sea    56.865   -164.065
      South Pacific    -0.001   -154.919
Southern California    34.308   -120.814


In [18]:
# ──────────────────────────────────────────────
# Re-train models (quick — needed for panels D, E, F)
# ──────────────────────────────────────────────

FEATURE_COLS = [
    "latitude", "longitude", "sst_in_situ_mean",
    "sat_sst_mean", "sat_sst_std", "sat_sst_min", "sat_sst_max",
    "sat_sst_median", "sat_sst_closest",
    "sat_chla_mean", "sat_chla_std", "sat_chla_min", "sat_chla_max",
    "sat_chla_median", "sat_chla_closest", "chla_days_offset",
]
TARGET = "pco2_mean"

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=df["location"]
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLS, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURE_COLS, index=X_test.index)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        random_state=42, n_jobs=-1),
}

results = []
predictions = {}

for name, model in models.items():
    if "Linear" in name:
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    results.append({"Model": name, "RMSE": round(rmse, 1), "MAE": round(mae, 1), "R2": round(r2, 3)})
    predictions[name] = y_pred
    print(f"{name:25s}  RMSE={rmse:.1f}  MAE={mae:.1f}  R²={r2:.3f}")

results_df = pd.DataFrame(results)
best_name = results_df.loc[results_df["RMSE"].idxmin(), "Model"]
print(f"\nBest model: {best_name}")

Linear Regression          RMSE=83.3  MAE=55.6  R²=0.281
Random Forest              RMSE=71.8  MAE=43.6  R²=0.465

Best model: Random Forest


In [19]:
# ── Save ────────────────────────────────────────────────────
# Ensure the output directory exists before saving the figure
PLOT_DIR.mkdir(parents=True, exist_ok=True)
save_path = PLOT_DIR / "summary_figure_6panel.png"
fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\nSaved → {save_path}")

C:\Users\Owner\AppData\Local\Temp\ipykernel_8656\40199740.py:5: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor="white")
C:\Users\Owner\AppData\Local\Temp\ipykernel_8656\40199740.py:5: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor="white")



Saved → C:\Users\Owner\OneDrive - UW\Desktop\Grad School\coursework\machine learning\ML2026_Orrand\final project deliverables\summary_figure_6panel.png
